# Local HybridQA workflow
CPU-only data inspection, preprocessing, sanity checks, analysis, and loading downloaded policy weights. Full neural training is intentionally absent.

In [ ]:
from pathlib import Path
import json

from data_pipeline import audit_dataset, load_config, preprocess_hybridqa, load_jsonl

config = load_config('config.yaml')
config['paths']

## 1. Audit official data

In [ ]:
RUN_FULL_AUDIT = False  # enable after both official repositories are present
if RUN_FULL_AUDIT:
    audit = audit_dataset(config, inspect_tables=True)
    print(json.dumps(audit, indent=2)[:5000])

## 2. Preprocess
Start with a small subset. Set `PREPROCESS_LIMIT=None` only after inspection succeeds.

In [ ]:
RUN_PREPROCESSING = False
PREPROCESS_LIMIT = 100
if RUN_PREPROCESSING:
    manifest = preprocess_hybridqa(config, max_examples=PREPROCESS_LIMIT)
    print(json.dumps(manifest, indent=2))

## 3. CPU sanity checks
These checks do not train on HybridQA.

In [ ]:
RUN_SANITY_CHECKS = False
if RUN_SANITY_CHECKS:
    from training import sanity_check_training_math
    sanity_check_training_math()
    print('Sanity checks passed.')

## 4. Analyze downloaded predictions

In [ ]:
from evaluation import aggregate_result_directory
result_dir = Path(config['paths']['output_dir']) / 'results'
prediction_files = sorted(result_dir.glob('*.jsonl')) if result_dir.exists() else []
if prediction_files:
    all_scores = aggregate_result_directory(result_dir)
    print(json.dumps(all_scores['summary'], indent=2))
else:
    print('No downloaded Colab predictions yet.')

## 5. Run downloaded model
Loads final selector weights on CPU. It never starts supervised or PPO training.

In [ ]:
RUN_DOWNLOADED_MODEL = False
if RUN_DOWNLOADED_MODEL:
    import yaml
    from models import build_actor_critic, load_policy_weights
    from training import verify_inference_bundle

    bundle = Path(config['paths']['output_dir']) / 'final_bundle'
    verify_inference_bundle(bundle)
    with (bundle / 'policy_config.yaml').open(encoding='utf-8') as handle:
        policy_config = yaml.safe_load(handle)
    policy = build_actor_critic(policy_config)
    load_policy_weights(policy, bundle / 'policy_state.pt', device='cpu')
    print('Final policy loaded on CPU. Use evaluation.policy_select(...) for an example.')